In [ ]:
import os, sys, time
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT", "/kaggle/working/ML-for-Wireless-Security-Threats-Identification"))
sys.path.insert(0, str(ROOT))
RESULTS_DIRECTORY, SAVED_MODEL_DIRECTORY = Path("results"), Path("saved_model")
RESULTS_DIRECTORY.mkdir(exist_ok=True)
SAVED_MODEL_DIRECTORY.mkdir(exist_ok=True)
import numpy as np
from shared.config import LABELS, SEED
from shared.data_loader import load_splits
from shared.evaluator import measure_latency_milliseconds, save_json, save_results
from shared.tracer import build_alerts, save_bundle
from sklearn.ensemble import RandomForestClassifier

In [ ]:
splits = load_splits()
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
start_time = time.perf_counter()
model.fit(splits["features_train"], splits["labels_train"])
train_time = time.perf_counter() - start_time

In [ ]:
predictions = model.predict(splits["features_test"])
latency = measure_latency_milliseconds(model.predict, splits["features_test"])
metrics = save_results(RESULTS_DIRECTORY, "random_forest", splits["labels_test"], predictions, train_time, latency,
                       (splits["features_train"].columns, model.feature_importances_))
save_bundle(SAVED_MODEL_DIRECTORY / "model.pkl", model, splits["schema"])

In [ ]:
from shared.tracer import ScikitLearnPredictor
probabilities = ScikitLearnPredictor(model, splits["schema"]).predict_proba(splits["features_test"])
alerts = build_alerts(splits["metadata_test"], probabilities.argmax(axis=1), probabilities.max(axis=1))
save_json(alerts, RESULTS_DIRECTORY / "trace_results.json")
print(metrics)
print(len(alerts), "incidents")